# 第3回 演習：汎化性能

## 今日の分析目標

**新しい日でも当たるモデルにしたい。**

前回の重回帰は R²=0.80 でしたが、それは学習に使ったデータでの自己採点でした。この演習では、データを訓練用とテスト用に分けて「新しいデータでの実力」を自分の手で測ります。TODOに取り組みながら、最後の「目標に答えられたか」で振り返りましょう。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
import japanize_matplotlib

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
DATA_DIR = 'https://raw.githubusercontent.com/k0heiun0/applied_exercise/main/shared/data'   # データはこのリポジトリから読み込む
df = pd.read_csv(f'{DATA_DIR}/bike_day.csv')
feat = ['temp','atemp','hum','windspeed','season','yr',
        'mnth','holiday','weekday','workingday','weathersit']
X = df[feat].values
y = df['cnt'].values
print(f'データサイズ: {X.shape}')

## 1. 分割なしで評価（NG例）

まず、あとで「やってはいけない」と分かる評価の典型を、あえて先に実際にやってみます。全データで学習し、その同じ全データで評価すると——

### 深掘り：なぜ「自己採点」は当てにならないのか

上のセルは、全データで学習し**その同じ全データ**で採点しました（R²=0.8002）。線形回帰では控えめな数字ですが、この測り方には原理的な問題があります。ここを腹落ちさせると、この演習の残り全部が「なぜそうするのか」まで見えてきます。

**訓練誤差と汎化誤差は別物**。学習に使ったデータの上での誤差を訓練誤差、まだ見ていない未知のデータの上での誤差を汎化誤差と呼びます。モデルを実際に使う場面では、相手はいつも新しいデータ（まだ来ていない日）です。手元にある日の利用台数はもう分かっているので、予測する意味がありません。だから本当に知りたいのは、つねに汎化誤差のほう。訓練誤差がどれだけ小さくても、汎化誤差が大きいモデルは役に立ちません。

**自己採点が甘くなる仕組み**。モデルは訓練データの答えをすでに見て、それに合うように係数を決めています。その同じデータで採点すれば、モデルは「自分が答えを覚えた問題」を解いているだけ——過去問の答えを丸暗記した学生が、まったく同じ過去問で満点を取るのと同じです。だから訓練データ上の成績は、原理的に**楽観の側へ偏ります**。これは運や偶然ではなく、測り方そのものに埋め込まれた偏りです。

**なぜ今回は差が小さいのか**。線形回帰は11変数と素直で、データは731日ぶんと十分にあります。自由度の低いモデルは、そもそもノイズまで覚え込む余地が小さいので、後で測る本番（テストR²）と自己採点（0.8002）がたまたま近くなります。ただしこれは線形回帰だからの話。**モデルが複雑になるほど、自己採点は楽観へ大きく振れます**——それを3節の多項式で目に見える形にします。だから鉄則は変わりません。**必ず、学習に使っていないデータで評価する**。

**「当てはまりの物差し」R² のおさらい**。R²（決定係数）は、目的変数のばらつきのうちモデルが説明できた割合で、1に近いほどよく当たっています。同じ R² でも、訓練データで測れば**訓練R²（見かけの性能）**、未知データで測れば**汎化性能の見積もり**——同じ計算式でも、どのデータで測るかで意味がまるで変わることに注意してください。

**複雑なモデルほど自己採点が甘くなる理由**。モデルに与える自由度（調整できるつまみの数）が多いほど、訓練データの細部——本質ではないたまたまのノイズ——にまで合わせ込めます。合わせ込んだぶん訓練R²は上がりますが、それは実力ではなく暗記。だから自由度の大きいモデルほど、自己採点と本番の落差は開きます。線形回帰でこの落差が小さかったのは「たまたま良いモデルだった」からではなく、「自由度が低くて暗記しにくいモデルだった」から、と理解しておくと応用が利きます。


In [ ]:
model_all = LinearRegression().fit(X, y)
r2_no_split = r2_score(y, model_all.predict(X))
print(f'全データ学習・全データ評価 → R²={r2_no_split:.4f}')
print('これは学習データでの自己採点（線形回帰では控えめだが、複雑なモデルほど楽観に振れる）')

### TODO①：train_test_split でデータを分割する

`train_test_split` を自分で書いて、データを訓練8割・テスト2割に分けてください。分け方を毎回同じにするため `random_state=42` を指定し、訓練とテストの件数を表示しましょう。

### 深掘り：なぜ「わざとデータを隠す」のか（分割の直感）

汎化誤差は「未知のデータでの誤差」。でも未知のデータは、定義からして手元にありません。まだ見ていないデータでの成績を、どうやって測ればいいのでしょうか。ここで使う工夫が、**手持ちのデータをわざと二つに分ける**ことです。一部を学習用（訓練）に使い、残りは学習に一切見せずに封をして取っておき、採点のときだけ開ける——この取っておいた側が「未知のフリ」をするテストデータです。学習にテストを混ぜた瞬間、未知のフリが崩れて自己採点に逆戻りするので、**テストは最後の答え合わせまで絶対に触らない**のが鉄則です。

**なぜランダムに分けるのか**。このデータは日付順に並んでいます。単純に先頭から8割・末尾2割で切ると、テストが特定の季節や年だけに偏り、評価が歪みます。`train_test_split` はくじ引きのように行を混ぜてから分けるので、訓練とテストが同じような顔ぶれになります。

**`random_state=42` の役割**。その「くじ引き」は乱数で決まるため、指定しないと実行のたびに分け方が変わり、結果も揺れます。`random_state` に数を固定すると、誰がいつ実行しても同じ分割になる——**再現性のための合言葉**です（42 という値そのものに意味はなく、単なる固定の目印）。

**割合の綱引き**。テスト2割で、訓練584日・テスト147日に分かれます。テストを増やすほど採点は安定しますが、そのぶん学習に回せるデータが減ってモデルが弱くなります。逆にテストが小さすぎると、採点自体が数件の運で揺れます。定番の 8:2 や 7:3 は、この綱引きの落としどころです。

**分割は「未知」の代役にすぎない**。テストデータは、あくまで手元にある既知のデータを未知に見立てた**代役**です。だから一つ暗黙の前提があります——テストデータが、これから相手にする本番のデータと**同じ素性（同じ母集団）**であること。もし本番だけ事情が違えば（たとえば新しい駐輪場、需要構造の変化など）、どれだけ公正に分割してもテストR²は本番を言い当てられません。「テストで良かった＝本番でも安心」が成り立つのは、両者が似た世界から来ているとき、という但し書きを頭の隅に置いてください。

**なお回帰では層化しない**。分類問題では、まれなクラスがテストに偏らないよう割合をそろえて分ける「層化抽出」を使いますが、今回の回帰では素直にシャッフルするだけで足ります。分割の作法も、扱う問題によって少しずつ変わることだけ覚えておきましょう。


In [ ]:
# TODO: train_test_split を使って、X と y を訓練8割・テスト2割に分割し、それぞれの件数を表示してください
# ヒント: 戻り値は「訓練用X・テスト用X・訓練用y・テスト用y」の4つ。分け方を固定する引数（毎回同じ分割にするためのもの）も指定しましょう
...

## 2. 訓練R²とテストR²を比べる

TODO①の答え合わせも兼ねて、同じ分割（`random_state=42`）で「過去問の点数」と「本番の点数」を測ります。

### 深掘り：訓練R²とテストR²の差を、どう読むか

この節で並べる二つの数字——訓練R² 0.7911 と テストR² 0.8277——の**差**が、過学習を見抜くいちばん素直な物差しです。

**差の向きと意味**。ふつうは訓練R²のほうが高く（自分が勉強した過去問だから）、その差が大きいほど「過去問はできるのに本番で落ちる」＝過学習のサインです。ところが今回は差が **0.7911 − 0.8277 = −0.037** と、めずらしく**テストのほうが高く**なっています。これは何かがおかしいのではなく、線形回帰が過学習していないうえに、**たまたま今回のテスト147日が訓練より少し「素直な」顔ぶれだった**というだけのこと。1回の分割の結果は、この程度の幅で上下に揺れます。

**一つの数字を信じすぎない**。だから「テストR² 0.828 が出た、実力は0.83だ」と早合点するのは危険です。別の乱数で分ければ、0.72 くらいまで下がることもあります（4節の交差検証で実際に見ます）。単発のテストR²は「実力のひとつの目撃例」であって、確定値ではありません。差の**符号や小数第2位**に一喜一憂せず、「訓練と本番がだいたい同じ水準か」という**大づかみ**で読むのが正しい態度です。両者が近いこと自体が、過学習していない健全なサインでした。

**自己採点との差も見ておく**。あわせて、分割なしの自己採点 0.8002 とテストR² 0.8277 の差（約 −0.027）も確認しておきましょう。今回はこれも小さい。線形回帰だからですが、複雑なモデルではこの差が大きく開きます——次節でそれを目にします。

**健全な差の目安**。ざっくりした読み方はこうです——訓練R²とテストR²が**どちらも高くて近い**なら理想（今回がこれ）。訓練R²は高いのにテストR²だけ**大きく低い**なら過学習。両方とも**低い**なら、そもそもモデルが単純すぎる未学習を疑います。差そのものより、「二つの水準の組み合わせ」で状態を診断するのがコツです。

**負の差は「狙えない」**。今回テストがわずかに上回りましたが、これを「良いこと」として狙いにいくことはできません。どちらが上に出るかは分割の運まかせで、こちらから操作できないからです。もし操作できてしまったら、それはテストを覗いて調整している——つまり5節で扱うデータリークが起きている合図です。


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression().fit(X_tr, y_tr)
r2_tr = r2_score(y_tr, model.predict(X_tr))
r2_te = r2_score(y_te, model.predict(X_te))
print(f'訓練R²  : {r2_tr:.4f}')
print(f'テストR²: {r2_te:.4f}')

### TODO②：訓練とテストの差を計算する

訓練R²とテストR²の差を計算して表示してください。この差が小さいほど「過去問と本番の落差が小さい」＝過学習していないサインです。分割なしのR²（`r2_no_split`）とテストR²の差も見てみましょう。

In [ ]:
# TODO: 「過去問と本番の落差」を数字にしてみましょう
# ヒント: 訓練R²とテストR²の引き算。自己採点（r2_no_split）とテストR²の間でも同じ計算をしてみる
...

## 3. 過学習を目で見る（多項式回帰）

気温（temp）1変数だけを使い、多項式の次数を上げてモデルを複雑にしていくと何が起きるかを観察します。なお `Pipeline` は、ここでは便利のため先取りで使うだけです（本格的には第9回で学びます）。まずは次数ごとの当てはまりを図で確認します。

### 深掘り：過学習の古典曲線と、その先で何が起こるか

ここがこの回のいちばんの山場です。気温1変数の多項式で次数（＝モデルの複雑さ）を上げると、当てはまりの線が波打っていくのは図で見たとおり。ではその裏で、**訓練の成績とテストの成績はどう動くのか**。TODO③で数字にすると、次のようになります（TODO③を正しく解いたときの実行値）。

| 次数 | 訓練R² | テストR² | ようす |
|---:|---:|---:|---|
| 1 | 0.3881 | 0.4037 | 単純すぎ（やや未学習） |
| 3 | 0.4720 | 0.4137 | テストがいちばん良い＝谷底 |
| 8 | 0.4769 | 0.4076 | テストは頭打ち〜微減 |
| 15 | 0.4863 | 0.4057 | 訓練だけ伸び、テストは離れる |

**古典的な二本の曲線**。訓練R²は次数とともに **0.388 → 0.472 → 0.477 → 0.486 と上がり続けます**。複雑にすればするほど過去問に合わせ込めるからで、これはどこまでも下がる訓練「誤差」の裏返しです。いっぽうテストR²は3次でピーク（0.4137）を打ったあと、8次・15次と**じわじわ下がって**いきます。誤差で描けば、テスト誤差は途中でいったん下がってから上がる**U字**。「訓練は下がり続けるのに、テストは途中から悪くなる」——これが過学習の古典的な絵で、今日ぜひ持ち帰ってほしい形です。訓練誤差だけを頼りに選ぶと、いちばん複雑な最悪のモデルを選んでしまう、という警告でもあります。

**なぜこうなるのか（骨子だけ）**。予測の外し（テスト誤差）は、大きく二つの原因に分けて考えられます。ひとつはモデルが単純すぎて本当の関係を表現しきれない**偏り（バイアス）**、もうひとつはデータの偶然のノイズにまで振り回される**ばらつき（バリアンス）**です。次数を上げるとバイアスは減りますが、バリアンスは増えます。低次では前者が、高次では後者が支配的になり、その**足し合わせ**が途中で底を打つ——これがU字の中身です。厳密な分解（バイアス・バリアンス分解）は lesson_MVA『汎化』回に譲りますが、「単純すぎても複雑すぎても損、ちょうどよい複雑さがある」という感覚がここでの収穫です。

**正直な但し書き**。今回のU字は、テスト側の悪化がごく浅いことに気づいたはずです。理由は、気温1変数の15次でもパラメータは16個で、訓練データ584日に対して**まだ圧倒的に少ない**から。過学習の激しい崩れは、パラメータをデータ量に近づけるほど鋭くなります。今回はU字の**左半分をゆるやかに歩いている**段階、と読むのが正確です。

**その先で何が起こるか——第二の谷（double descent）への予告**。では想像してみてください。「複雑にするほどテストは悪くなる」なら、複雑さをどこまでも上げ続けたら、テスト誤差はどこまでも悪化し続けるのでしょうか。じつは、話には続きがあります。パラメータ数がデータ数に追いつくあたり（**補間閾値**）でモデルは訓練データを誤差ゼロで通し切れるようになり、テスト誤差はここで**いったん最悪の山**を迎えます。ところが、**そこを越えてさらにパラメータを増やす**と、テスト誤差が**もう一度下がりはじめる**ことが知られています。U字の山の向こうに、**第二の下り坂**が現れるのです。この現象を **double descent（二重降下）** と呼びます。

ここでは名前と「U字の先にもう一段ある」という事実だけ、予告に留めます。なぜ起こるのか、どんな設定で顔を出すのか——その詳しい仕組みは、**第7回（正則化）の演習**でじっくり扱います。ひとまず、「複雑さを上げる＝過学習へ近づく」という今日の直感は、無限にまっすぐ延長できるわけではない、という含みだけ覚えておいてください。理論的な扱いは lesson_MVA『汎化』回もあわせてどうぞ。


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

X1d_tr = X_tr[:, [0]]   # temp（気温）だけ
X1d_te = X_te[:, [0]]

degrees = [1, 3, 8, 15]
fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True)
x_plot = np.linspace(X1d_tr.min(), X1d_tr.max(), 200).reshape(-1, 1)
for ax, d in zip(axes, degrees):
    pipe = Pipeline([('poly', PolynomialFeatures(d)), ('lr', LinearRegression())])
    pipe.fit(X1d_tr, y_tr)
    ax.scatter(X1d_tr, y_tr, s=6, alpha=0.2, color='#0066cc')
    ax.plot(x_plot, np.clip(pipe.predict(x_plot), -2000, 11000), color='#e63946', lw=2)
    ax.set_title(f'{d}次')
    ax.set_ylim(0, 9000)
plt.suptitle('多項式の次数と当てはまり（気温→利用台数）')
plt.tight_layout()
plt.show()

### TODO③：次数を変えて過学習を観察する

`degrees = [1, 3, 8, 15]` のそれぞれについて、訓練R²とテストR²を計算して表示してください。次数を上げると訓練R²とテストR²はそれぞれどう動くでしょうか。

In [ ]:
# TODO: 次数ごとに訓練R²とテストR²を計算し、次数を上げると2つがどう動くか観察してください
# ヒント: for文で degrees の各次数を回し、その次数のモデルを学習 → 訓練とテストのそれぞれで R² を計算して表示
...

## 4. 交差検証で「分け方の運」をならす

1回の分割は運でぶれます。5分割の交差検証で、平均とばらつきを確認しましょう。データは日付順なので、`shuffle=True` で並び順の偏りを除きます。

### 深掘り：交差検証はなぜ「平均±ばらつき」で見るのか

3節までで、1回のテストR²は分け方の運で揺れる、と繰り返してきました。交差検証（クロスバリデーション）は、その運を**ならして**測るための道具です。

**5分割の仕組み**。データを5つのかたまりに分け、1つ目をテスト・残り4つを訓練にして採点、次は2つ目をテスト……とテスト役を順に交代させます。こうすると全データがちょうど一度ずつテストに使われ、5回ぶんの採点が手に入ります。1回きりの運に頼らず、5つの目撃例で判断できるわけです。

**`shuffle=True` の意味**。このデータは日付順なので、そのまま5分割すると各かたまりが特定の季節・年に偏ります。`shuffle=True`（と `random_state`）で行をよく混ぜてから分け、どのかたまりも似た顔ぶれになるようにしています。

**実際の数字が語ること**。5回の結果は `[0.828, 0.746, 0.813, 0.722, 0.808]`。同じモデル・同じデータでも、分け方だけで **0.72〜0.83** と1割近く揺れています。平均は **0.783**、ばらつき（標準偏差）は **0.042**。ここで2節の「テストR² 0.828」を思い出してください。あれは5回のうち**いちばん高い回**とほぼ同じ値——つまり単発では**運のいい目撃例**を見ていたわけです。交差検証を通すと、実力の中心は0.83ではなく **0.78あたり**だと分かります。2節で「一つの数字を信じすぎない」と言った理由が、ここで数字になって見えます。

**なぜ平均とばらつきをセットで見るのか**。平均は、より信頼できる実力の推定値。ばらつきは、その推定の**不確かさ**そのものです。ばらつきが小さければ「どう分けても安定して当たる」、大きければ「分け方しだいで結果が動く危ういモデル」。二つのモデルを比べるとき、平均だけ見て「こっちが0.01高いから勝ち」と決めるのは、ばらつき0.042の前では早計だと分かります。**数字ひとつでなく、幅で比べる**——これが安全なモデル選びの作法です。

**コスト**。交差検証は学習を5回繰り返すぶん計算が重くなります。迷ったらまず Train/Test、モデルを比較・選択する大事な場面では交差検証、と使い分けます。データが何百万件もある場面では、1回の分割でも十分に安定します。

**交差検証も万能ではない**。5回の採点は、訓練データを大きく共有しています（どの回も全体の8割が訓練）。そのため5つのスコアは完全には独立でなく、標準偏差はほんとうの揺れをやや**小さめに**見積もる傾向があります。それでも、1回の分割よりはるかに信頼できる目安です。分割数 K を増やすほど採点は安定しますが計算は重くなり、極端に K をデータ件数まで増やしたものが Leave-One-Out（1件だけテスト）です。時系列やグループ構造があるデータでは、専用の分け方（時系列分割・グループ分割）を使う——これも「問題に合わせて分割を選ぶ」話の延長です。


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), X, y, cv=kf, scoring='r2')
print(f'各回のR²: {scores.round(3)}')
print(f'平均R²: {scores.mean():.3f} / ばらつき(標準偏差): {scores.std():.3f}')

## 5. データリークを再現する

前処理（標準化）を分割の前にやってしまう、ありがちなリークをわざと再現し、正しい順序と比べます。

### 深掘り：データリーク——いちばん怖い、気づけない過ち

ここまで丁寧に分割しても、**データリーク**があると評価は一気に台無しになります。リークとは、テストデータの情報がこっそり学習側に混入すること。未知のフリが崩れ、また自己採点に逆戻りしてしまう罠です。厄介なのは、**丁寧に前処理をした人ほど、気づかずにやりがち**な点です。

**前処理の順序によるリーク**。上のセルの標準化を例にします。標準化は「平均を引いて標準偏差で割る」操作。ここで**全データで平均・標準偏差を計算してから分割**すると、その平均・標準偏差には**テストの値も混ざって**います。モデルは本来知らないはずのテストの情報を、前処理を通じて間接的に覗いてしまう。正しい順序は必ず、**先に分割 → 訓練だけで平均・標準偏差を学習（fit）→ テストにはそれを適用（transform）するだけ**。テストは最後まで「未知」を貫きます。

**なぜ今回は差が出ないのか（そして油断してはいけない理由）**。上の表では「リークあり」も「正しい順序」も **R²=0.8277 と完全に同じ**でした。これは線形回帰に**標準化しても予測が変わらない**という性質があるからで、リークが無害だったわけではありません。第7回で学ぶ**正則化つき回帰**（罰金が係数のスケールに依存）や、距離を使う kNN などでは、標準化の順序が結果を**直接**動かします。「今回は同じだった」を一般化せず、**どの手法でも安全な正しい順序を、機械的に守る**のが正解です。

**もっと怖いターゲットリーク**。前処理よりたちが悪いのが、**答えそのものに近い情報**を説明変数に入れてしまうケースです。この自転車データには、じつは「登録利用者数」と「臨時利用者数」という列もあります。もしこれらを特徴量に入れたら——**二つを足すと利用台数そのもの**。答えを直接見て予測しているのと同じで、評価はほぼ満点に化けます。でも本番で当てたいのは「まだ来ていない明日」。その日の登録・臨時利用者数は、まだ誰にも分かりません。だから本番でぼろぼろに崩れます。**精度が良すぎるときほどリークを疑う**——うますぎる話には裏がある、が鉄則です。

**判断のたった一つの問い**。特徴量を一つずつ、こう問うてください——「**この情報は、予測したい時点で本当に手に入るか？**」。結果が出たあとに初めて分かる情報は、説明変数に入れてはいけません。

**見るたびに漏れる、隠れたリーク**。最後にもう一つ。テストR²を見ながら「次数を変えては測り直し、いちばん良かったものを選ぶ」——これも実は隠れたリークです。テストを何度も覗いて調整すると、テストの情報が少しずつモデル選びに漏れ込み、そのテストR²はもう当てになりません。テストは本番、何度も覗かない。手綱の調整には、テストではなく**交差検証**や検証用データを使うのが正しい作法です。

**順序を仕組みで守る**。前処理の順序を毎回手で守るのはミスのもとです。scikit-learn の**パイプライン**は「標準化→学習」をひとまとめにして、交差検証の各分割の中で自動的に正しい順序（訓練だけで fit）を守ってくれます。仕組みで縛ってしまえば、うっかりリークを構造的に防げます。この道具は第9回の演習で本格的に扱います。

**落とし穴の三点まとめ**。①訓練データで評価して喜ぶ——必ずテストで測る。②前処理を分割の前にやる——リークの典型。③テストを何度も見て調整する——見るたびに少しずつ漏れる。この三つを避け、「テストは最後の答え合わせ」を徹底することが、正しく測る力の土台になります。


In [ ]:
# NG: 全データで標準化してから分割（テストの平均・分散が漏れる）
scaler_leak = StandardScaler()
X_scaled_all = scaler_leak.fit_transform(X)
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_scaled_all, y, test_size=0.2, random_state=42)
r2_leak = r2_score(y_te_l, LinearRegression().fit(X_tr_l, y_tr_l).predict(X_te_l))

# OK: 先に分割 → 訓練だけで前処理を学習 → テストには適用のみ
scaler_ok = StandardScaler()
X_tr_s = scaler_ok.fit_transform(X_tr)
X_te_s = scaler_ok.transform(X_te)
r2_ok = r2_score(y_te, LinearRegression().fit(X_tr_s, y_tr).predict(X_te_s))

results = pd.DataFrame({'手法': ['分割なし（自己採点）', 'Train/Test分割', 'リークあり', '正しい順序'],
                        'R²': [r2_no_split, r2_te, r2_leak, r2_ok]})
print(results.to_string(index=False))
print('※ 線形回帰には標準化で予測が変わらない性質があるため今回は差が出ない。')
print('   ただし多くの手法では前処理の順序が結果を直接左右する。手順は常に正しく守ること')

## 目標に答えられたか

- 今日の目標は「新しい日でも当たるモデルにしたい」でした
- TODO②の結果を見て、「新しいデータでの実力」として信じるべき数字はどれでしょうか？
- TODO②で、訓練R²とテストR²の差はほとんどありませんでした。これは何を意味するでしょうか？
- TODO③で、次数を上げると訓練R²とテストR²はどう動きましたか？どの次数を選ぶべきでしょうか？
- R²という物差しだけで「当たる」と言い切ってよいでしょうか？（→第5回で評価指標を深めます）

## 課題（提出）

**提出するもの**: 応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

2節では `random_state=42` で1回だけ分割し、テストR²を測りました。今度は分け方を変えて、同じことを5回くり返します。
`random_state` を 0, 1, 2, 3, 4 の5通りに変えて `X`, `y` を訓練8割・テスト2割に分割し、それぞれで `LinearRegression` を学習して、テストR²を5つ表示してください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

2節のセルの分割・学習・テストR² の3行を for 文の中に入れ、`random_state=42` の部分を for の変数にします。テストR²はリストに貯めておくと応用②で使えます。

</details>


### 応用②（判断）

応用①で得た5つのテストR²のうち、最大と最小の**差**は幾つですか。丸めていない5つの値で差を計算し、その差を小数第2位に丸めて答えてください（例: 0.12）。先に各値を丸めてから引くと答えがずれることがあります。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

リストを `np.array` にすれば `.max() - .min()` で差が出ます。`f'{差:.2f}'` と書くと小数第2位までの表示になります。

</details>


### 応用③（解釈）

2節では1回の分割でテストR²が 0.8277 でした。この1回の値だけを根拠に「このモデルの実力は R²=0.83」と報告してよいでしょうか。
応用①の5つの値の幅と、4節の交差検証の結果（平均とばらつき）を根拠に、自転車シェアの運営担当者に向けて3行で書いてください。


（ここに3行程度で書く）


## 発展（任意）

### TimeSeriesSplit（日付順のデータで未来を覗かない）

bike_day は 2011年1月1日から2012年12月31日まで、日付順に並んだ時系列データです。2節や4節のランダムな分割は行をよく混ぜてから分けるので、「2012年の日を訓練に使って2011年の日を当てる」ことが普通に起きます。本番では逆です。手元にあるのは過去だけで、当てたいのはまだ来ていない未来です。未来の情報を訓練に混ぜた評価は、5節のリークと同じで成績を甘く見せます。

売上・気象・電力など時間順のデータは世の中に多く、この落とし穴はどこにでもあります。`TimeSeriesSplit` は「過去の区間で学び、その直後の区間で採点する」をくり返す分割で、この順序を守ってくれます。


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

kf  = KFold(n_splits=5, shuffle=True, random_state=42)   # 4節と同じランダム分割
tss = TimeSeriesSplit(n_splits=5)                        # 過去 → 未来の順で分割
s_kf  = cross_val_score(LinearRegression(), X, y, cv=kf,  scoring='r2')
s_tss = cross_val_score(LinearRegression(), X, y, cv=tss, scoring='r2')
print(f'ランダム分割: {s_kf.round(3)}  平均={s_kf.mean():.3f}')
print(f'時系列分割  : {s_tss.round(3)}  平均={s_tss.mean():.3f}')


In [ ]:
# 時系列分割の各回で「どの期間で学び、どの期間を当てたか」を見る
for i, (tr, te) in enumerate(tss.split(X), start=1):
    print(f'{i}回目: 訓練 {df["dteday"].iloc[tr[0]]}〜{df["dteday"].iloc[tr[-1]]} ({len(tr)}日)'
          f' → テスト {df["dteday"].iloc[te[0]]}〜{df["dteday"].iloc[te[-1]]} ({len(te)}日)'
          f'  R²={s_tss[i-1]:.3f}')


ランダム分割の平均は 0.783 でした。時系列分割では5回の R² が `[-4.526, 0.249, -0.425, -0.182, 0.600]`、平均 −0.857 と、まるで別のモデルのような数字になります。R² が負というのは「そのテスト区間の実際の平均値をそのまま予測にするより外している」という意味です。比べる相手が「まだ来ていない未来の平均」という、本番では知りようのない厳しい基準なので、時系列の評価では負の値が出ます。

1回目が −4.5 と極端に悪いのは、訓練が 2011年1〜5月の126日だけだからです。夏を一度も見ていないモデルに、5〜9月の利用台数を当てさせています。訓練で見た `temp` の最大は 0.63 なのに、テストには 0.85 まで出てきます。見たことのない高温域へ直線を伸ばして予測した結果が −4.5 です。

3回目（2012年1〜5月を当てる）が −0.425 と再び負になるのは、`yr`（年）のせいです。訓練368日のうち 2012年は1月1〜3日の3日だけで、テストは全て 2012年です。2011年から2012年への利用台数の増加を、3日分の情報からは学べません。

全体としては訓練期間が延びるにつれて改善し、5回目（2012年9月1日までの610日で学び、残りの4か月を当てる）で R²=0.600 です。本番にいちばん近いのはこの5回目で、それでもランダム分割の 0.78 には届きません。ランダム分割の 0.78 は「未来を覗いた」甘い成績で、過去だけから未来を当てる実力はそれよりかなり低い、というのがこのデータの正直な姿です。

試すなら `n_splits` を 8 に変えて、区間が短くなると各回の R² がどう動くか見てください。
